In [1]:
pip install choix numpy pandas


[notice] A new release of pip is available: 24.3.1 -> 26.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

import pandas as pd
import json
import ast 

import numpy as np
import choix

In [3]:
def load_dicts_to_df(
    directory: str | Path,
    pattern: str = "*",
) -> pd.DataFrame:
    directory = Path(directory)
    if not directory.is_dir():
        raise NotADirectoryError(directory)

    rows: list[dict[str, Any]] = []
    for path in sorted(directory.glob(pattern)):
        if not path.is_file() or path.name.startswith("."):
            continue
        suffix = path.suffix.lower()
        data = None
        if suffix == ".json":
            with path.open(encoding="utf-8") as f:
                data = json.load(f)
        else:
            text = path.read_text(encoding="utf-8")
            data = ast.literal_eval(text)

        if not isinstance(data, dict):
            raise TypeError(f"{path} does not contain a dict (got {type(data).__name__})")
        rows.append(data)

    return pd.json_normalize(rows)

In [4]:
# git clone https://huggingface.co/spaces/taagarwa/coding-agent-leaderboard
df = load_dicts_to_df("/home/eje/git/coding-agent-leaderboard/results", pattern="*.json")

In [5]:
df.columns

Index(['benchmark.name', 'benchmark.repo', 'benchmark.num_tasks',
       'benchmark.url', 'harness.name', 'harness.skills', 'harness.is_oss',
       'harness.url', 'model.name', 'model.repo', 'model.is_oss',
       'model.num_params', 'model.precision', 'model.url', 'environment.name',
       'environment.config.name', 'environment.url', 'metrics.n_tasks',
       'metrics.n_errors', 'metrics.score', 'metrics.n_input_tokens',
       'metrics.n_cache_tokens', 'metrics.n_output_tokens',
       'metrics.n_total_tokens', 'metrics.agent_time_seconds',
       'metrics.total_time_seconds', 'metrics.cost_usd',
       'metrics.mean_input_tokens_per_task',
       'metrics.mean_cache_tokens_per_task',
       'metrics.mean_output_tokens_per_task', 'metrics.mean_tokens_per_task',
       'metrics.mean_cost_usd_per_task',
       'metrics.mean_total_time_seconds_per_task',
       'metrics.mean_agent_time_seconds_per_task', 'environment.config.path',
       'environment.config.version', 'environment.con

In [6]:
df = df[['model.name', 'harness.name', 'metrics.score', 'metrics.mean_cost_usd_per_task', 'metrics.mean_tokens_per_task', 'benchmark.name']]
df

,model.name,harness.name,metrics.score,metrics.mean_cost_usd_per_task,metrics.mean_tokens_per_task,benchmark.name
0,Gemma4-31B-FP8,Claude Code,0.417,0.57,1288269.0,SWE-Bench Pro -- Ansible
1,Gemma4-31B-FP8,OpenCode,0.417,0.20,1055516.0,SWE-Bench Pro -- Ansible
2,Gemma4-31B-FP8,Pi,0.469,0.17,833841.0,SWE-Bench Pro -- Ansible
3,Gemma4-31B-FP8,Claude Code,0.612,0.21,1311796.0,SWE-Bench Verified
4,Gemma4-31B-FP8,OpenCode,0.606,0.11,1059493.0,SWE-Bench Verified
5,Gemma4-31B-FP8,Pi,0.574,0.08,763314.0,SWE-Bench Verified
6,Nemotron-3-Super-120B-NVFP4,Claude Code,0.224,0.11,2210591.0,RH SWE-Bench
7,Nemotron-3-Super-120B-NVFP4,OpenCode,0.308,0.10,2367765.0,RH SWE-Bench
8,Nemotron-3-Super-120B-NVFP4,Pi,0.216,0.09,2176734.0,RH SWE-Bench
9,Nemotron-3-Super-120B-NVFP4,Claude Code,0.432,0.28,4259014.0,SWE-Bench Pro -- Ansible


In [7]:
df = df.dropna()
df = df.loc[df['metrics.mean_tokens_per_task'] > 0]
df = df.reset_index(drop=True)
df

,model.name,harness.name,metrics.score,metrics.mean_cost_usd_per_task,metrics.mean_tokens_per_task,benchmark.name
0,Gemma4-31B-FP8,Claude Code,0.417,0.57,1288269.0,SWE-Bench Pro -- Ansible
1,Gemma4-31B-FP8,OpenCode,0.417,0.20,1055516.0,SWE-Bench Pro -- Ansible
2,Gemma4-31B-FP8,Pi,0.469,0.17,833841.0,SWE-Bench Pro -- Ansible
3,Gemma4-31B-FP8,Claude Code,0.612,0.21,1311796.0,SWE-Bench Verified
4,Gemma4-31B-FP8,OpenCode,0.606,0.11,1059493.0,SWE-Bench Verified
5,Gemma4-31B-FP8,Pi,0.574,0.08,763314.0,SWE-Bench Verified
6,Nemotron-3-Super-120B-NVFP4,Claude Code,0.224,0.11,2210591.0,RH SWE-Bench
7,Nemotron-3-Super-120B-NVFP4,OpenCode,0.308,0.10,2367765.0,RH SWE-Bench
8,Nemotron-3-Super-120B-NVFP4,Pi,0.216,0.09,2176734.0,RH SWE-Bench
9,Nemotron-3-Super-120B-NVFP4,Claude Code,0.432,0.28,4259014.0,SWE-Bench Pro -- Ansible


In [8]:
from typing import Any
from itertools import groupby
def prepare_ranking_data(df: pd.DataFrame,
                         catcol: str | list[str],
                         metcol: str,
                         descending: bool = False,
                         eqvcol: str | list[str] = []) -> tuple[list[tuple[int, int]], list[Any], list[float]]:
    ndata = df.shape[0]
    if ndata < 2:
        raise ValueError("Not enough data to prepare ranking comparisons")
    catcol = catcol if isinstance(catcol, list) else [catcol]
    eqvcol = eqvcol if isinstance(eqvcol, list) else [eqvcol]
    ncat = len(catcol)
    neqv = len(eqvcol)
    if ncat < 1:
        raise ValueError("No category column provided")
    tcols = catcol + eqvcol + [metcol]
    t = list(df[tcols].itertuples(index=False, name=None))
    metvals = [x[-1] for x in t]
    if ncat > 1:
        catvals = [x[:ncat] for x in t]
    else:
        # single category column: categories are just the values of the column
        catvals = [x[0] for x in t]
    if neqv > 0:
        # we will only compare items in the same equivalence group
        eqvvals = [x[ncat:ncat+neqv] for x in t]
    else:
        # by default all items are treated as one equivalence group
        eqvvals = [None] * ndata
    # map item categories to unique integers
    umap = dict([(y,x) for x,y in enumerate(sorted(set(catvals)))])
    compvals = [(umap[c],m,e) for c,m,e in zip(catvals, metvals, eqvvals)]
    comps = []
    for i in range(ndata):
        ic, im, ie = compvals[i]
        for j in range(i):
            jc, jm, je = compvals[j]
            if ie != je:
                # ignore items with different equivalence values
                continue
            if im == jm:
                # ignore items with same metric value
                continue
            # pairs are always of form (winner, loser)
            iwin = im < jm if descending else im > jm
            if iwin:
                comps.append((ic, jc))
            else:
                comps.append((jc, ic))
    return comps, sorted(umap.keys())

In [9]:
comps, cats = prepare_ranking_data(df, 'harness.name', 'metrics.score', eqvcol='benchmark.name')
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank:2d}. {params[idx]:+.3f}  {cats[idx]}")


Ranking:
   1. +1.453  Codex
   2. +0.205  Qwen Code
   3. -0.240  Claude Code
   4. -0.700  Pi
   5. -0.718  OpenCode


In [10]:
comps, cats = prepare_ranking_data(df, 'model.name', 'metrics.score', eqvcol='benchmark.name')
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank:2d}. {params[idx]:+.3f}  {cats[idx]}")

Ranking:
   1. +9.263  Opus 4.8
   2. +7.603  Opus 4.6
   3. +4.773  GPT 5.5 - high
   4. +1.453  Sonnet 4.6
   5. -3.849  Qwen3.6-35B-A3B-NVFP4
   6. -4.479  Gemma4-31B-FP8
   7. -6.623  Nemotron-3-Super-120B-NVFP4
   8. -8.141  Mistral-Small-4-119B-2603-NVFP4


In [11]:
comps, cats = prepare_ranking_data(df, ['model.name', 'harness.name'], 'metrics.score', eqvcol='benchmark.name')
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank:2d}. {params[idx]:+.3f}  {cats[idx]}")

Ranking:
   1. +10.114  ('Opus 4.8', 'OpenCode')
   2. +10.114  ('Opus 4.8', 'Claude Code')
   3. +9.001  ('Opus 4.6', 'Claude Code')
   4. +7.221  ('GPT 5.5 - high', 'Codex')
   5. +5.397  ('Sonnet 4.6', 'Claude Code')
   6. +3.654  ('Qwen3.6-35B-A3B-NVFP4', 'Pi')
   7. +0.359  ('Qwen3.6-35B-A3B-NVFP4', 'Claude Code')
   8. +0.359  ('Qwen3.6-35B-A3B-NVFP4', 'Qwen Code')
   9. -0.583  ('Gemma4-31B-FP8', 'Pi')
  10. -1.599  ('Gemma4-31B-FP8', 'Claude Code')
  11. -2.121  ('Gemma4-31B-FP8', 'OpenCode')
  12. -3.366  ('Nemotron-3-Super-120B-NVFP4', 'Claude Code')
  13. -3.974  ('Qwen3.6-35B-A3B-NVFP4', 'OpenCode')
  14. -4.838  ('Nemotron-3-Super-120B-NVFP4', 'Pi')
  15. -5.006  ('Mistral-Small-4-119B-2603-NVFP4', 'OpenCode')
  16. -5.592  ('Nemotron-3-Super-120B-NVFP4', 'OpenCode')
  17. -6.982  ('Mistral-Small-4-119B-2603-NVFP4', 'Pi')
  18. -12.156  ('Mistral-Small-4-119B-2603-NVFP4', 'Claude Code')


In [12]:
comps, cats = prepare_ranking_data(df, ['model.name', 'harness.name'], 'metrics.mean_cost_usd_per_task', eqvcol='benchmark.name', descending=True)
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank:2d}. {params[idx]:+.3f}  {cats[idx]}")

Ranking:
   1. +12.329  ('Mistral-Small-4-119B-2603-NVFP4', 'Pi')
   2. +10.847  ('Mistral-Small-4-119B-2603-NVFP4', 'Claude Code')
   3. +8.644  ('Mistral-Small-4-119B-2603-NVFP4', 'OpenCode')
   4. +5.266  ('Qwen3.6-35B-A3B-NVFP4', 'Claude Code')
   5. +5.266  ('Qwen3.6-35B-A3B-NVFP4', 'Qwen Code')
   6. +5.002  ('Qwen3.6-35B-A3B-NVFP4', 'OpenCode')
   7. +4.290  ('Nemotron-3-Super-120B-NVFP4', 'OpenCode')
   8. +3.711  ('Qwen3.6-35B-A3B-NVFP4', 'Pi')
   9. +2.929  ('Gemma4-31B-FP8', 'Pi')
  10. +2.820  ('Nemotron-3-Super-120B-NVFP4', 'Pi')
  11. +0.861  ('Gemma4-31B-FP8', 'OpenCode')
  12. +0.116  ('Nemotron-3-Super-120B-NVFP4', 'Claude Code')
  13. -3.334  ('Gemma4-31B-FP8', 'Claude Code')
  14. -6.486  ('Opus 4.8', 'OpenCode')
  15. -9.165  ('Sonnet 4.6', 'Claude Code')
  16. -12.605  ('Opus 4.8', 'Claude Code')
  17. -13.165  ('Opus 4.6', 'Claude Code')
  18. -17.325  ('GPT 5.5 - high', 'Codex')


In [13]:
comps, cats = prepare_ranking_data(df, ['model.name', 'harness.name'], 'metrics.mean_tokens_per_task', eqvcol='benchmark.name', descending=True)
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank:2d}. {params[idx]:+.3f}  {cats[idx]}")

Ranking:
   1. +6.403  ('Mistral-Small-4-119B-2603-NVFP4', 'Pi')
   2. +5.982  ('Gemma4-31B-FP8', 'Pi')
   3. +4.905  ('Mistral-Small-4-119B-2603-NVFP4', 'Claude Code')
   4. +4.905  ('Mistral-Small-4-119B-2603-NVFP4', 'OpenCode')
   5. +4.905  ('Gemma4-31B-FP8', 'OpenCode')
   6. +3.510  ('Gemma4-31B-FP8', 'Claude Code')
   7. +3.116  ('Qwen3.6-35B-A3B-NVFP4', 'OpenCode')
   8. +2.674  ('Qwen3.6-35B-A3B-NVFP4', 'Qwen Code')
   9. -1.100  ('Opus 4.8', 'OpenCode')
  10. -2.801  ('Opus 4.8', 'Claude Code')
  11. -3.388  ('Qwen3.6-35B-A3B-NVFP4', 'Claude Code')
  12. -3.590  ('Nemotron-3-Super-120B-NVFP4', 'OpenCode')
  13. -3.665  ('GPT 5.5 - high', 'Codex')
  14. -3.799  ('Sonnet 4.6', 'Claude Code')
  15. -4.013  ('Nemotron-3-Super-120B-NVFP4', 'Claude Code')
  16. -4.225  ('Qwen3.6-35B-A3B-NVFP4', 'Pi')
  17. -4.684  ('Nemotron-3-Super-120B-NVFP4', 'Pi')
  18. -5.136  ('Opus 4.6', 'Claude Code')


In [14]:
comps, cats = prepare_ranking_data(df, 'model.name' , 'metrics.score', eqvcol='benchmark.name')
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank:2d}. {params[idx]:+.3f}  {cats[idx]}")


Ranking:
   1. +9.263  Opus 4.8
   2. +7.603  Opus 4.6
   3. +4.773  GPT 5.5 - high
   4. +1.453  Sonnet 4.6
   5. -3.849  Qwen3.6-35B-A3B-NVFP4
   6. -4.479  Gemma4-31B-FP8
   7. -6.623  Nemotron-3-Super-120B-NVFP4
   8. -8.141  Mistral-Small-4-119B-2603-NVFP4


In [15]:
H = lambda p: float(sum(-p * np.log2(p)))
xmean = lambda x: float((np.sum(x)-1.0)/(len(x)-1))
x = [[H(choix.probabilities([cc, c], params)) for c in range(len(cats))] for cc in range(len(cats))]
[xmean(e) for e in x]


[0.08866754533374495,
 0.23042870338207674,
 0.13604276389191036,
 0.2129215281787499,
 0.13804828396719065,
 0.10387348343779713,
 0.2007336976683312,
 0.04602335900786362]